Our package provides data access in a Python programming environment.

Here, we will start a Clustering analysis for the Pancreatic ductal adenocarcinoma (pdac).

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

# from gpnotebook.tools.standard_imports import *
import os, re,sys
import yaml
import pandas as pd
import numpy as np


In [2]:
# project_dir = r"/Users/yingweihu/Documents/GitHub/glycoproteinnotebook-private/data/v1/projects/PDAC_P_PDC000271"
project_dir = r"/Users/yingweihu/Documents/GitHub/glycoproteinnotebook-private/data/v1/projects/LSCC_P_PDC000232"
data_dir = os.path.join(project_dir,"matrix")
meta_dir = os.path.join(project_dir,"meta")
job_dir = os.path.join(project_dir,"precomputed","cluster")
if not os.path.exists(job_dir):
    os.mkdir(job_dir)

In [3]:
data_path = os.path.join(data_dir, "DIG_nglycoform-peptide_matrix-abundances-MD_norm.tsv")
data_df = pd.read_csv(data_path,sep="\t", index_col = [0,1,2,3])
data_df

Intensity.Reference  \
Site                                               Gene     Sequence                        Glycan                            
ENSP00000226382@158                                PHOX2B   AAAAAAAAAKNGSSGK                N5H4F2S1G0            15.651139   
                                                                                            N7H4F2S1G0            13.892873   
                                                                                            N7H5F0S0G0            14.868719   
                                                                                            N7H8F6S1G0            14.444237   
                                                            AAAAAAAAAKNGSSGKK               N4H5F1S4G0            15.793260   
...                                                                                                                     ...   
ENSP00000376793@267;ENSP00000451119@49             SERPINA3 YTGNASALFILPDQDK                N7H8F2S1G0            11.416615   
                                                            YTGNASALFILPDQDKMEEVEAMLLPETLK  N4H5F3S2G0            11.399708   
                                                            YTGNASALFILPDQDKMEEVEAMLLPETLKR N5H8F2S1G0            11.026810   
ENSP00000261590@458                                DSG2     YVQNGTYTVK                      N7H7F3S1G0            10.302947   
ENSP00000413130@124;ENSP00000438497@212;ENSP000... GNS      YYNYTLSINGK                     N3H4F1S2G0            13.324512   

                                                                                                        C3L-02665_T_01  \
Site                                               Gene     Sequence                        Glycan                       
ENSP00000226382@158                                PHOX2B   AAAAAAAAAKNGSSGK                N5H4F2S1G0       15.651139   
                                                                                            N7H4F2S1G0       13.892873   
                                                                                            N7H5F0S0G0       14.868719   
                                                                                            N7H8F6S1G0       14.444237   
                                                            AAAAAAAAAKNGSSGKK               N4H5F1S4G0       15.793260   
...                                                                                                                ...   
ENSP00000376793@267;ENSP00000451119@49             SERPINA3 YTGNASALFILPDQDK                N7H8F2S1G0             NaN   
                                                            YTGNASALFILPDQDKMEEVEAMLLPETLK  N4H5F3S2G0             NaN   
                                                            YTGNASALFILPDQDKMEEVEAMLLPETLKR N5H8F2S1G0             NaN   
ENSP00000261590@458                                DSG2     YVQNGTYTVK                      N7H7F3S1G0             NaN   
ENSP00000413130@124;ENSP00000438497@212;ENSP000... GNS      YYNYTLSINGK                     N3H4F1S2G0             NaN   

                                                                                                        C3L-02665_N_01  \
Site                                               Gene     Sequence                        Glycan                       
ENSP00000226382@158                                PHOX2B   AAAAAAAAAKNGSSGK                N5H4F2S1G0       14.193687   
                                                                                            N7H4F2S1G0       11.525209   
                                                                                            N7H5F0S0G0       15.733382   
                                                                                            N7H8F6S1G0       14.570092   
                                                            AAAAAAAAAKNGSSGKK               N4H5F1S4G0             NaN   
...        

In [4]:
meta_path= os.path.join(meta_dir, "LSCC_meta.txt")
meta_df = pd.read_csv(meta_path,sep="\t",header=[0,1])
meta_df

,case_id,Age,Sex,Tumor_Size_cm,Histologic_Grade,Tumor_necrosis,Path_Stage_pT,Path_Stage_pN,Stage,BMI,Tobacco_smoking_history,KEAP1_mutation,FAT1_mutation,ARID1A_mutation,PIK3CA_mutation,TP53_mutation,CDKN2A_mutation,NOTCH1_mutation
,data_type,CON,BIN,CON,ORD,BIN,ORD,ORD,ORD,CON,ORD,BIN,BIN,BIN,BIN,BIN,BIN,BIN
0,C3L-00081,61,Female,5.0,G2 Moderately differentiated,Not identified,pT2,pN1,Stage II,32.29,non-smoker,1,0,0,0,1,0,0
1,C3L-00415,64,Female,6.5,G3 Poorly differentiated,Not identified,pT2,pN0,Stage II,27.64,past smoker,0,0,1,0,1,0,0
2,C3L-00445,74,Male,5.0,G2 Moderately differentiated,Present,pT2,pN0,Stage II,29.98,past smoker,0,1,0,0,1,1,0
3,C3L-00568,74,Female,4.7,G2 Moderately differentiated,Not identified,pT2,pN0,Stage I,28.52,past smoker,1,0,0,0,1,0,0
4,C3L-00603,72,Male,3.0,G3 Poorly differentiated,Not identified,pT1,pN0,Stage I,35.68,past smoker,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,C3N-03886,60,Male,4.2,NaN,Present,pT2,pN0,Stage I,29.99,non-smoker,0,0,0,0,1,0,1
104,C3N-04124,62,Male,3.5,G3 Poorly differentiated,Present,pT2,pN0,Stage I,21.01,current smoker,0,0,0,0,1,0,0
105,C3N-04127,63,Male,5.0,G2 Moderately differentiated,Not identified,pT2,pN2,Stage III,21.16,current smoker,0,0,0,0,1,0,0


In [5]:
meta_cols = ['case_id','Sex','Stage']
meta2 = meta_df.loc[:,meta_cols]
meta2.columns = ['Sample.ID'] + meta_cols[1:]
meta2

,Sample.ID,Sex,Stage
0,C3L-00081,Female,Stage II
1,C3L-00415,Female,Stage II
2,C3L-00445,Male,Stage II
3,C3L-00568,Female,Stage I
4,C3L-00603,Male,Stage I
...,...,...,...
103,C3N-03886,Male,Stage I
104,C3N-04124,Male,Stage I
105,C3N-04127,Male,Stage III
106,C3N-04155,Male,Stage I


In [6]:
meta2.head(17)

,Sample.ID,Sex,Stage
0,C3L-00081,Female,Stage II
1,C3L-00415,Female,Stage II
2,C3L-00445,Male,Stage II
3,C3L-00568,Female,Stage I
4,C3L-00603,Male,Stage I
5,C3L-00904,Male,Stage II
6,C3L-00923,Female,Stage I
7,C3L-00927,Male,Stage III
8,C3L-00965,Male,Stage II
9,C3L-00993,Male,Stage III


In [7]:
head_cols = ['Site', 'Gene', 'Sequence', 'Glycan', 'Intensity.Reference']
samples = [i for i in data_df.columns.values if i not in head_cols]
samples = [i for i in samples if i.split('_')[0] in list(meta2['Sample.ID']) and i.split('_')[1] == 'T']
len(samples)

108

In [8]:
rows = []
for sample in samples:
    key = sample.split('_')[0]
    row = meta2[meta2['Sample.ID']==key].iloc[0]
    row['Sample.ID'] = sample
    rows.append(row)
meta3 = pd.DataFrame(rows)

In [9]:
meta3.head(17)

,Sample.ID,Sex,Stage
36,C3L-02665_T_01,Male,Stage II
14,C3L-01663_T_01,Female,Stage III
84,C3N-02575_T_01,Male,Stage I
25,C3L-02546_T_01,Male,Stage I
8,C3L-00965_T_01,Male,Stage II
40,C3L-02963_T_02,Male,Stage I
107,C3N-04162_T_02,Male,Stage II
31,C3L-02646_T_02,Male,Stage III
71,C3N-02285_T_02,Female,Stage II
99,C3N-03875_T_02,Male,Stage I


In [10]:
meta3 = meta3.replace(np.nan,'NA')

In [11]:
top_ann_data_path = os.path.join(job_dir,'top_ann_data.tsv')
meta3.to_csv(top_ann_data_path, sep="\t", index=False)

Top annotation settings.

In [12]:

top_ann_settings = {
    'Sex': {
        'Male': 'blue',
        'Female': 'red',
        'NA': 'grey',
    },
    'Stage': {
        'Stage I': 'blue',
        'Stage II': 'green',
        'Stage III': 'orange',
        'Stage IV': 'red',
        'NA': 'grey'
    },

}
top_ann_settings_path = os.path.join(job_dir,'top_ann_settings.yml')
with open(top_ann_settings_path,'w') as f:
    yaml.dump(top_ann_settings,f,default_flow_style=False)

In [13]:
data_df.head(2)

Intensity.Reference  \
Site                Gene   Sequence         Glycan                            
ENSP00000226382@158 PHOX2B AAAAAAAAAKNGSSGK N5H4F2S1G0            15.651139   
                                            N7H4F2S1G0            13.892873   

                                                        C3L-02665_T_01  \
Site                Gene   Sequence         Glycan                       
ENSP00000226382@158 PHOX2B AAAAAAAAAKNGSSGK N5H4F2S1G0       15.651139   
                                            N7H4F2S1G0       13.892873   

                                                        C3L-02665_N_01  \
Site                Gene   Sequence         Glycan                       
ENSP00000226382@158 PHOX2B AAAAAAAAAKNGSSGK N5H4F2S1G0       14.193687   
                                            N7H4F2S1G0       11.525209   

                                                        C3L-01663_T_01  \
Site                Gene   Sequence         Glycan                       
ENSP00000226382@158 PHOX2B AAAAAAAAAKNGSSGK N5H4F2S1G0       14.520324   
                                            N7H4F2S1G0       13.859168   

                                                        C3L-01663_N_01  \
Site                Gene   Sequence         Glycan                       
ENSP00000226382@158 PHOX2B AAAAAAAAAKNGSSGK N5H4F2S1G0       13.954894   
                                            N7H4F2S1G0       12.413791   

                                                        C3N-02575_T_01  \
Site                Gene   Sequence         Glycan                       
ENSP00000226382@158 PHOX2B AAAAAAAAAKNGSSGK N5H4F2S1G0       15.470535   
                                            N7H4F2S1G0       14.869817   

                                                        C3N-02575_N_01  \
Site                Gene   Sequence         Glycan                       
ENSP00000226382@158 PHOX2B AAAAAAAAAKNGSSGK N5H4F2S1G0       13.779659   
                                            N7H4F2S1G0       11.831512   

                                                        C3L-02546_T_01  \
Site                Gene   Sequence         Glycan                       
ENSP00000226382@158 PHOX2B AAAAAAAAAKNGSSGK N5H4F2S1G0       14.807007   
                                            N7H4F2S1G0       13.559205   

                                                        C3L-02546_N_01  \
Site                Gene   Sequence         Glycan                       
ENSP00000226382@158 PHOX2B AAAAAAAAAKNGSSGK N5H4F2S1G0       13.857714   
                                            N7H4F2S1G0       12.929426   

                                                        C3L-00965_T_01  ...  \
Site                Gene   Sequence         Glycan                      ...   
ENSP00000226382@158 PHOX2B AAAAAAAAAKNGSSGK N5H4F2S1G0       15.085593  ...   
                                            N7H4F2S1G0       13.984816  ...   

                                                        C3N-01892_N_22  \
Site                Gene   Sequence         Glycan                       
ENSP00000226382@158 PHOX2B AAAAAAAAAKNGSSGK N5H4F2S1G0             NaN   
                                            N7H4F2S1G0             NaN   

                                                        C3L-00603_T_22  \
Site                Gene   Sequence         Glycan                       
ENSP00000226382@158 PHOX2B AAAAAAAAAKNGSSGK N5H4F2S1G0             NaN   
                                            N7H4F2S1G0             NaN   

                                                        C3L-00603_N_22  \
Site                Gene   Sequence         Glycan                       
ENSP00000226382@158 PHOX2B AAAAAAAAAKNGSSGK N5H4F2S1G0             NaN   
                                            N7H4F2S1G0             NaN   

                                                        C3L-03965_T_22  \
Site                Gene   Sequence         Glycan                       
ENSP000

In [14]:
data_df.shape

(65747, 243)

In [15]:
samples = meta3['Sample.ID'].to_list()

In [16]:
len(samples)

108

In [17]:
df2 = data_df.loc[:,samples].dropna()

In [18]:
df2.shape

(680, 108)

In [19]:
from scipy.stats import variation
rows = []
for index,row in df2.iterrows():
    rows.append([variation([np.power(2,i) for i in list(row)])])
cv_df = pd.DataFrame(rows,columns=['cv'],index= df2.index)

glycopeptides = cv_df[cv_df['cv']>0.25].index

data2 = df2[df2.index.isin(glycopeptides)]
glycopeptides =  [f'{site}@{gene}@{seq}@{glycan}' for site,gene,seq,glycan in glycopeptides]
data2.index = glycopeptides
tumor_expression_path = os.path.join(job_dir,'expression_data.tsv')
data2.to_csv(tumor_expression_path,sep='\t',index=True)

In [20]:
data2.shape

(587, 108)

Extract tumor samples from glycopeptide expression data based on pathological status,

calculates the coefficient of variation (CV) for each glycopeptide, selects glycopeptides with CV greater than 0.25.

Map glcopeptides with cv>0.25 in tumor patients with glycan type.

In [21]:
import re,os, sys

def decide_glycan_type(g):
    m = re.finditer("([A-Z])([\d]+)", g)
    y = [(i.group(1), int(i.group(2))) for i in m]
    d = dict(y)
    glycan_type = "Other"
    if d["N"] == 2 and d["H"] >= 5 and d["F"] == 0 and d["S"] == 0 and d["G"] == 0:
        glycan_type = "HM"
    elif d["N"] >= 2 and d["H"] >= 3 and d["F"] > 0 and d["S"] == 0:
        glycan_type = "only_F"
    elif d["N"] >= 2 and d["H"] >= 3 and d["S"] > 0 and d["F"] == 0:
        glycan_type = "only_S"
    elif d["N"] >= 2 and d["H"] >= 3 and d["S"] > 0 and d["F"] > 0:
        glycan_type = "F+S"
    return glycan_type


In [22]:
# left annotation
# from gpnotebook.tools.glycan import decide_glycan_type

glycan_type_map = dict(zip(glycopeptides,[decide_glycan_type(i) for i in glycopeptides]))
  
left_ann_data_path =  os.path.join(job_dir,'left_annotation_data.tsv')
rows = []
for i in glycan_type_map:
    rows.append([i,glycan_type_map[i]])
left_ann_data = pd.DataFrame(rows,columns=['Glycopeptide','GlycanType'])
left_ann_data.to_csv(left_ann_data_path,sep="\t",index=False)

In [23]:
left_ann_data

,Glycopeptide,GlycanType
0,ENSP00000262776@541@LGALS3BP@AAIPSALDTNSSK@N4H...,F+S
1,ENSP00000262776@541@LGALS3BP@AAIPSALDTNSSK@N4H...,F+S
2,ENSP00000262776@541@LGALS3BP@AAIPSALDTNSSK@N5H...,F+S
3,ENSP00000262776@541@LGALS3BP@AAIPSALDTNSSK@N5H...,F+S
4,ENSP00000273784@166;ENSP00000393887@165@AHSG@A...,F+S
...,...,...
582,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,only_S
583,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,F+S
584,ENSP00000376793@267;ENSP00000451119@49@SERPINA...,F+S
585,ENSP00000261590@458@DSG2@YVQNGTYTVK@N4H5F1S1G0,F+S


Map glycan types with colors.

In [24]:

# left annotation settings, including color, order
left_ann_settings_path = os.path.join(job_dir,'left_annotation_settings.yml')
left_ann_settings = {
    "glycan_type_index" :{
    "HM": 1,
    "only_F":2,
    "only_S":3,
    "F+S":4,
    "Other":5
    },
    "glycan_type_color" : {
        "HM": 'green',
    "only_F": 'red',
    "only_S": 'purple',
    "F+S": 'orange',
    "Other": 'grey'
}
}
with open(left_ann_settings_path,'w') as f:
    yaml.dump(left_ann_settings,f,default_flow_style=False)
    

Parameters for NMF clustering.

In [25]:
nmf_parameters_path = os.path.join(job_dir, 'nmf_parameters.yml')
nmf_parameters = {
    'k_range': {
        'min': 3,
        'max': 5,
    },
    'test':{
        'nruns': 50
    },
    'opt_k':{
        'nruns': 500,
        'predefined': 0,
        'value': 4,
        'feature_prob': 0.8
    }
}
with open(nmf_parameters_path,'w') as f:
    yaml.dump(nmf_parameters,f,default_flow_style=False)

Generate a YAML configuration file (nmf_configs.yml) containing paths to various data required for NMF clustering.

In [26]:
config_data = {
    'input': {
        'expression_data': tumor_expression_path,
        'left_annotation_data': left_ann_data_path ,
        'left_annotation_settings': left_ann_settings_path,
        'top_annotation_data': top_ann_data_path,
        'top_annotatin_settings': top_ann_settings_path,
        'nmf_parameters': nmf_parameters_path
    },
    'output':{
        'out_dir': job_dir
    }
}
nmf_configs_path = os.path.join(job_dir,'nmf_configs.yml')
with open(nmf_configs_path,'w') as f:
    yaml.dump(config_data,f,default_flow_style=False)